In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from sklearn.neighbors import BallTree
import os
import folium

In [ ]:
df0 = pd.read_csv("../data/raw/AIS_2018_09_29.csv")

row_num = 50000
df0.head(row_num).to_csv(f"../data/processed/ais_sample{row_num}.csv", index=False)

In [ ]:
# LOADING AND SORTING THE DATA #

filepath = f'../data/processed/ais_sample{row_num}.csv'

df = pd.read_csv(filepath) 


df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"]) # Convert to datetime

df = df.sort_values(['MMSI', 'BaseDateTime'])

def normalise(series):

    series = series.fillna(0)

    min_val = series.min()
    max_val = series.max()

    if max_val - min_val == 0:
        return pd.Series(0.0, index=series.index)

    return (series - min_val) / (max_val - min_val)


In [ ]:
# ANALYZING AIS TRANSMISSION GAPS #

df['time_delta'] = df.groupby('MMSI')['BaseDateTime'].diff().dt.total_seconds()/60  # Time delta in minutes

df['minor_gap_flag'] = (df['time_delta'] > 10 ) & (df['time_delta'] <= 30)
df['major_gap_flag'] = (df['time_delta'] > 30 ) & (df['time_delta'] <= 180)
df['dark_gap_flag'] = (df['time_delta'] > 180 ) 

gap_summary = df.groupby('MMSI').agg(
    minor_gaps=('minor_gap_flag','sum'),
    major_gaps=('major_gap_flag','sum'),
    dark_gaps=('dark_gap_flag','sum'),
    avg_gap=('time_delta','mean')
)

gap_summary['gap_count'] = (
    gap_summary['minor_gaps'] +
    gap_summary['major_gaps'] +
    gap_summary['dark_gaps']
)

if gap_summary['gap_count'].sum() > 0:
    plt.figure(figsize=(10,6))
    sns.histplot(gap_summary[gap_summary["gap_count"] > 0]["gap_count"], bins=20) # Plot only vessels with at least one gap
    plt.title("Distribution of AIS Transmission Gaps")
    plt.xlabel(f"Number of total AIS gaps per vessel")
    plt.ylabel("Count")
    plt.show()

if gap_summary['minor_gaps'].sum() > 0:
    plt.figure(figsize=(10,6))
    sns.histplot(gap_summary[gap_summary["minor_gaps"] > 0]["minor_gaps"], bins=20, color = 'yellow') # Plot only vessels with at least one gap
    plt.title("Distribution of AIS Transmission Gaps")
    plt.xlabel(f"Number of minor AIS gaps per vessel")
    plt.ylabel("Count")
    plt.show()
    
if gap_summary['major_gaps'].sum() > 0:
    plt.figure(figsize=(10,6))
    sns.histplot(gap_summary[gap_summary["major_gaps"] > 0]["major_gaps"], bins=20, color = 'orange') # Plot only vessels with at least one gap
    plt.title("Distribution of AIS Transmission Gaps")
    plt.xlabel(f"Number of major AIS gaps per vessel")
    plt.ylabel("Count")
    plt.show()

if gap_summary['dark_gaps'].sum() > 0:
    plt.figure(figsize=(10,6))
    sns.histplot(gap_summary[gap_summary["dark_gaps"] > 0]["dark_gaps"], bins=20, color = 'red') # Plot only vessels with at least one gap
    plt.title("Distribution of AIS Transmission Gaps")
    plt.xlabel(f"Number of dark AIS gaps per vessel")
    plt.ylabel("Count")
    plt.show()

In [ ]:
# ======================================
# Ship-to-Ship (STS) Detection
# Optimised Version
# ======================================

import numpy as np
import pandas as pd

print("Starting STS detection")

# ----------------------------------
# Ensure timestamp format
# ----------------------------------

df["timestamp"] = pd.to_datetime(df["BaseDateTime"])

print("Total AIS points:", len(df))
print("Unique vessels:", df["MMSI"].nunique())

# ----------------------------------
# Filter low-speed vessels
# (STS requires vessels nearly stationary)
# ----------------------------------

print("\nFiltering low-speed AIS points (SOG < 2 knots)")

sts_candidates = df[df["SOG"] < 2].copy()

print("Candidate AIS points:", len(sts_candidates))

# ----------------------------------
# Create time windows
# ----------------------------------

print("\nCreating 30-minute time windows")

sts_candidates["time_bin"] = sts_candidates["timestamp"].dt.floor("30min")

# ----------------------------------
# Create geographic grid
# ----------------------------------

print("Creating spatial grid")

GRID_SIZE = 0.05   # ~5km grid

sts_candidates["lat_bin"] = (sts_candidates["LAT"] / GRID_SIZE).astype(int)
sts_candidates["lon_bin"] = (sts_candidates["LON"] / GRID_SIZE).astype(int)

# ----------------------------------
# Group by space-time cells
# ----------------------------------

groups = sts_candidates.groupby(["time_bin","lat_bin","lon_bin"])

print("Total space-time groups:", len(groups))

events = []

# ----------------------------------
# Iterate through groups
# ----------------------------------

for i, ((time_bin, lat_bin, lon_bin), subset) in enumerate(groups):

    if i % 500 == 0:
        print(f"Processing group {i}/{len(groups)}")

    # Skip groups without multiple vessels
    if subset["MMSI"].nunique() < 2:
        continue

    coords = subset[["LAT","LON"]].values
    mmsi = subset["MMSI"].values
    sog = subset["SOG"].values

    n = len(subset)

    for a in range(n):

        for b in range(a+1,n):

            if mmsi[a] == mmsi[b]:
                continue

            # Haversine distance
            lat1, lon1 = np.radians(coords[a])
            lat2, lon2 = np.radians(coords[b])

            dlat = lat2 - lat1
            dlon = lon2 - lon1

            h = (
                np.sin(dlat/2)**2 +
                np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
            )

            dist = 6371 * 2 * np.arcsin(np.sqrt(h))

            # STS proximity threshold
            if dist < 0.5:   # km

                events.append({
                    "MMSI1": mmsi[a],
                    "MMSI2": mmsi[b],
                    "time_bin": time_bin
                })


print("\nSTS scanning complete")
print("Total candidate encounters:", len(events))


# ----------------------------------
# Convert to dataframe
# ----------------------------------

sts_df = pd.DataFrame(events)

print("STS dataframe shape:", sts_df.shape)


# ----------------------------------
# Aggregate STS events per vessel
# ----------------------------------

if not sts_df.empty and "MMSI1" in sts_df.columns and "MMSI2" in sts_df.columns:
    sts_counts_1 = sts_df.groupby("MMSI1").size()
    sts_counts_2 = sts_df.groupby("MMSI2").size()
    sts_counts = sts_counts_1.add(sts_counts_2, fill_value=0)
    sts_counts = sts_counts.rename("STS_Count")
    # Ensure all vessels appear
    sts_counts = sts_counts.reindex(df["MMSI"].unique()).fillna(0)
else:
    # No events found, create empty counts for all vessels
    sts_counts = pd.Series(0, index=df["MMSI"].unique(), name="STS_Count")

print("\nSTS vessels detected:", (sts_counts > 0).sum())

print("STS detection finished")

In [ ]:
# # VISUALIZING STS ENCOUNTER NETWORK #

# import networkx as nx

# # Load the STS events from the progress CSV
# sts_progress_path = 'sts_events_progress.csv'
# sts_df = pd.read_csv(sts_progress_path)

# # Create a graph where nodes are vessels and edges represent potential STS encounters, weighted by the number of events
# G = nx.Graph()

# for _, row in sts_df.iterrows():
#     G.add_edge(int(row['MMSI1']), int(row['MMSI2']), weight=int(row['event_count']))

# plt.figure(figsize=(10,8))
# nx.draw(G, node_size=10, with_labels=False)
# plt.title("Potential Ship-to-Ship Encounter Network")
# plt.show()

In [ ]:
# LOITERING ANALYSIS #

v_lower = 2  # Speed threshold for loitering (in knots)

df['loiter_flag'] = df["SOG"] < v_lower

loiter = df[df['loiter_flag']].groupby('MMSI').size() # Count of low-speed records per vessel

total = df.groupby('MMSI').size() # Total records per vessel

loiter_ratio = (loiter / total).fillna(0) # Proportion of low-speed records per vessel

# Duration based loitering
loiter_duration = df[df['loiter_flag']].groupby('MMSI')['time_delta'].sum()


plt.figure(figsize=(10,6))
sns.histplot(loiter_ratio, bins=20)
plt.title("Distribution of Loitering Ratio")
plt.xlabel("Loitering Ratio (proportion of low-speed records)")
plt.show()

In [ ]:
# ROUTE IRREGULARITY #

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

df['lat_dff'] = df.groupby('MMSI')['LAT'].diff()
df['lon_dff'] = df.groupby('MMSI')['LON'].diff()

df['dist_km'] = haversine(
    df['LAT'],
    df['LON'],
    df.groupby('MMSI')['LAT'].shift(),
    df.groupby('MMSI')['LON'].shift()
)

route_irregularity = df.groupby('MMSI')['dist_km'].std().fillna(0)


In [ ]:
# Ensure vessel name column exists
df["VesselName"] = df["VesselName"].astype(str)

# Count unique vessel names per MMSI
name_counts = df.groupby("MMSI")["VesselName"].nunique()

name_changes = name_counts[name_counts > 1]

print("Vessels with multiple names associated with the same MMSI:")
print(name_changes)

def detect_name_change_events(df):

    df = df.sort_values(["MMSI","BaseDateTime"])

    df["previous_name"] = (
        df.groupby("MMSI")["VesselName"]
        .shift()
    )

    df["name_change_flag"] = (
        df["VesselName"] != df["previous_name"]
    )

    events = df[df["name_change_flag"]]

    return events[["MMSI","BaseDateTime","previous_name","VesselName"]]

name_change_events = detect_name_change_events(df)

name_changes_count = name_change_events.groupby("MMSI").size().rename("Name_Change_Count")



In [ ]:
# Map each vessel to its total STS event count (sum over all edges)
sts_counts_1 = sts_df.groupby("MMSI1").size()
sts_counts_2 = sts_df.groupby("MMSI2").size()
sts_counts = sts_counts_1.add(sts_counts_2, fill_value=0)
sts_counts = sts_counts.rename("STS_Count")
# Ensure all vessels appear
sts_counts = sts_counts.reindex(df["MMSI"].unique()).fillna(0)


vessels = df['MMSI'].unique()

indicators = pd.DataFrame(index=vessels)

indicators['AIS_Gap_Count'] = gap_summary['gap_count']
indicators['Loiter_Ratio'] = loiter_ratio
indicators['Route_Irregularity'] = route_irregularity
indicators['STS_Count'] = sts_counts
indicators['Name_Change_Count'] = name_changes_count

indicators = indicators.fillna(0)


# NORMALISE INDICATORS

indicators['gap_score'] = normalise(indicators['AIS_Gap_Count'])

indicators['loiter_score'] = normalise(indicators['Loiter_Ratio'])

indicators['route_score'] = normalise(indicators['Route_Irregularity'])

indicators['sts_score'] = np.log1p(indicators['STS_Count'])
indicators['sts_score'] = normalise(indicators['sts_score'])

indicators['name_change_score'] = normalise(indicators['Name_Change_Count'])

# WEIGHTED RISK SCORE

weights = {
    "gap":0.25,
    "loiter":0.20,
    "route":0.15,
    "sts":0.25,
    "name":0.15
}

indicators["Risk_Score"] = (
    weights["gap"]*indicators['gap_score'] +
    weights["loiter"]*indicators['loiter_score'] +
    weights["route"]*indicators['route_score'] +
    weights["sts"]*indicators['sts_score'] +
    weights["name"]*indicators['name_change_score']
)

print(indicators['Risk_Score'])
indicators.to_csv(f"../data/processed/vessel_risk_indicators{row_num}.csv")
indicators['Risk_Score'].describe()

In [ ]:
# Ensure Risk_Category column exists
def risk_category(score):
    if score < 0.3:
        return "Low"
    elif score < 0.6:
        return "Moderate"
    elif score < 0.8:
        return "High"
    else:
        return "Extreme"

if "Risk_Category" not in indicators.columns:
    indicators["Risk_Category"] = indicators["Risk_Score"].apply(risk_category)

plt.figure(figsize=(10,6))
colors = {
    "Low": "#4CAF50",
    "Moderate": "#FFC107",
    "High": "#FF5722",
    "Extreme": "#9C27B0"
}
for category, color in colors.items():
    sns.histplot(
        indicators[indicators["Risk_Category"] == category]["Risk_Score"],
        bins=20, 
        color=color, 
        label=category, 
        alpha=0.7
    )
plt.title("Distribution of Maritime Behaviour Risk Scores by Category")
plt.xlabel("Risk Score")
plt.legend(title="Risk Category")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

corr = indicators[[
    'gap_score',
    'loiter_score',
    'route_score',
    'sts_score',
    'name_change_score'
]].corr()

sns.heatmap(corr, annot=True, cmap="coolwarm")

plt.title("Indicator Correlation Matrix")
plt.show()

In [ ]:
!pip install folium

In [ ]:
import folium

m = folium.Map(location=[df['LAT'].mean(), df['LON'].mean()], zoom_start=2, tiles='CartoDB positron'
               )

In [ ]:
for mmsi, vessel in df.groupby('MMSI'):
    coords = list(zip(vessel['LAT'], vessel['LON'])) 

    if len(coords) > 2:

        folium.PolyLine(coords, color='blue', weight=1, opacity=0.4).add_to(m)

In [ ]:
for mmsi, row in indicators.iterrows():

    vessel_data = df[df['MMSI'] == mmsi]

    if len(vessel_data) == 0:
        continue

    last_point = vessel_data.iloc[-1]

    risk = row['Risk_Score']

    folium.CircleMarker(
        location=[last_point['LAT'], last_point['LON']],
        radius=3 + risk*10,
        color='red' if risk > 0.7 else 'orange',
        fill=True,
        fill_opacity=0.7,
        popup=f"MMSI:{mmsi} Risk:{round(risk,2)}"
    ).add_to(m)

# STS HOTSPOT LOCATIONS

if len(sts_df) > 0:

    sts_points = []

    for _, row in sts_df.iterrows():

        v1_data = df[df['MMSI'] == row['MMSI1']]
        v2_data = df[df['MMSI'] == row['MMSI2']]

        if v1_data.empty or v2_data.empty:
            continue

        v1 = v1_data.iloc[-1]
        v2 = v2_data.iloc[-1]

        lat_mid = (v1['LAT'] + v2['LAT'])/2
        lon_mid = (v1['LON'] + v2['LON'])/2

        sts_points.append((lat_mid, lon_mid))

    for lat, lon in sts_points:

        folium.CircleMarker(
            location=[lat, lon],
            radius=4,
            color='purple',
            fill=True
        ).add_to(m)

m


In [ ]:
# ======================================
# AIS Gap Distribution
# ======================================

plt.figure(figsize=(10,6))

plt.hist(gap_summary["gap_count"], bins=30)

plt.title("Distribution of AIS Signal Gaps per Vessel")

plt.xlabel("Number of AIS Gaps")
plt.ylabel("Number of Vessels")

plt.show()

In [ ]:
# ======================================
# Indicator Pairplot
# ======================================

import seaborn as sns

indicator_subset = indicators[
[
"gap_score",
"loiter_score",
"route_score",
"sts_score",
"name_change_score"
]
]

sns.pairplot(indicator_subset)

plt.show()

In [ ]:
# ======================================
# Risk Score Sensitivity Analysis
# ======================================

weights_range = [0.1,0.2,0.3,0.4,0.5]

results = []

for w in weights_range:

    temp_score = (
        w * indicators["gap_score"] +
        w * indicators["sts_score"] +
        (1-2*w) * (
            indicators["loiter_score"] +
            indicators["route_score"]
        ) / 2
    )

    results.append(temp_score.std())

plt.figure(figsize=(8,5))

plt.plot(weights_range, results, marker="o")

plt.title("Risk Score Sensitivity to Indicator Weighting")

plt.xlabel("Weight applied to AIS Gap & STS indicators")

plt.ylabel("Standard Deviation of Risk Scores")

plt.show()

In [ ]:
# ======================================
# Top Risk Vessels
# ======================================

top_vessels = indicators.sort_values(
    "Risk_Score",
    ascending=False
).head(10)

plt.figure(figsize=(10,6))

plt.barh(
    top_vessels.index.astype(str),
    top_vessels["Risk_Score"]
)

plt.title("Top 10 Highest Risk Vessels")

plt.xlabel("Risk Score")
plt.ylabel("Vessel MMSI")

plt.gca().invert_yaxis()

plt.show()

In [ ]:
# ======================================
# Vessel Risk Ranking Curve
# ======================================

sorted_scores = indicators["Risk_Score"].sort_values()

plt.figure(figsize=(10,6))

plt.plot(sorted_scores.values)

plt.title("Ranked Vessel Risk Scores")

plt.xlabel("Vessel Rank")
plt.ylabel("Risk Score")

plt.show()